In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
train_df = pd.read_csv("/content/drive/MyDrive/DSO530 Project/insurance_train2026.csv")
test_df = pd.read_csv("/content/drive/MyDrive/DSO530 Project/insurance_test2026.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())
display(test_df.head())

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/DSO530 Project/insurance_train2026.csv'

In [ ]:
# Basic structure check

train_df.info()
display(train_df.describe())

test_df.info()
display(test_df.describe())

##Data Preprocessing and Cleaning

In [ ]:
# Dropping X.1, the ID column
train_df.drop(columns=["X.1"], inplace=True, errors="ignore")
test_df.drop(columns=["X.1"], inplace=True, errors="ignore")

In [ ]:
# Formatting the date columns

date_cols = ["X.2", "X.3", "X.4", "X.5", "X.6"]

for col in date_cols:
    train_df[col] = pd.to_datetime(train_df[col], errors='coerce')
    test_df[col] = pd.to_datetime(test_df[col], errors='coerce')

In [ ]:
# Creating a simple policy duration feature

train_df["policy_duration"] = (train_df["X.4"] - train_df["X.2"]).dt.days
test_df["policy_duration"] = (test_df["X.4"] - test_df["X.2"]).dt.days

In [ ]:
# # Handling missing values
# # Fill numeric columns with the median.

# for col in train_df.columns:
#     if train_df[col].dtype != "object":
#         train_df[col] = train_df[col].fillna(train_df[col].median())

# for col in test_df.columns:
#     if test_df[col].dtype != "object":
#         test_df[col] = test_df[col].fillna(test_df[col].median())

In [ ]:
# # Fill categorical columns with the mode.

# for col in train_df.columns:
#     if train_df[col].dtype == "object":
#         train_df[col] = train_df[col].fillna(train_df[col].mode()[0])

# for col in test_df.columns:
#     if test_df[col].dtype == "object":
#         test_df[col] = test_df[col].fillna(test_df[col].mode()[0])

In [ ]:
# No duplicate rows exist but dropping them is good practice I suppose

train_df[train_df.duplicated()]

In [ ]:
# Removing duplicate rows

train_df.drop_duplicates(inplace=True)
test_df.drop_duplicates(inplace=True)

In [ ]:
# # Saving the cleaned datasets

# train_df.to_csv("insurance_train2026_cleaned.csv", index=False)
# test_df.to_csv("insurance_test2026_cleaned.csv", index=False)

# print("Train shape:", train_df.shape)
# print("Test shape:", test_df.shape)

# print("\nTrain preview:")
# display(train_df.head())

# print("\nTest preview:")
# display(test_df.head())

# print("\nTrain info:")
# train_df.info()

# print("\nTest info:")
# test_df.info()

In [ ]:
train_df.head()

In [ ]:
test_df.head()

##Creating Target Variables

In [ ]:
# Creating target variables
# LC = X.15 / X.16 if X.16 > 0, otherwise 0
# HALC = LC * X.18 if X.16 > 0, otherwise 0
# CS = 1 if number of claims is greater than 0, otherwise 0

train_df["LC"] = np.where(train_df["X.16"] > 0, train_df["X.15"] / train_df["X.16"], 0)

train_df["HALC"] = np.where(train_df["X.16"] > 0, train_df["LC"] * train_df["X.18"], 0)

train_df["CS"] = (train_df["X.16"] > 0).astype(int)

display(train_df[["X.15", "X.16", "X.18", "LC", "HALC", "CS"]].head())

In [ ]:
# train_df.isna().sum()

##EDA: Target Distributions

In [ ]:
# Original LC distribution

plt.figure()
sns.histplot(train_df["LC"], bins=50)
plt.title("Original Distribution of LC (Skewed)")
plt.xlabel("LC")
plt.ylabel("Count")
plt.show()

In [ ]:
# Original HALC distribution

plt.figure()
sns.histplot(train_df["HALC"], bins=50)
plt.title("Original Distribution of HALC (Skewed)")
plt.xlabel("HALC")
plt.ylabel("Count")
plt.show()

In [ ]:
# Log distribution of LC

plt.figure()
sns.histplot(np.log1p(train_df["LC"]), bins=50)
plt.title("Log Distribution of LC")
plt.xlabel("log(1 + LC)")
plt.ylabel("Count")
plt.show()

In [ ]:
# Log distribution of HALC

plt.figure()
sns.histplot(np.log1p(train_df["HALC"]), bins=50)
plt.title("Log Distribution of HALC")
plt.xlabel("log(1 + HALC)")
plt.ylabel("Count")
plt.show()

In [ ]:
# Log distribution of HALC

plt.figure()
sns.histplot(np.log1p(train_df["HALC"]), bins=50)
plt.title("Log Distribution of HALC")
plt.xlabel("log(1 + HALC)")
plt.ylabel("Count")
plt.show()

In [ ]:
# LC distribution for non-zero claims only

plt.figure()
sns.histplot(train_df[train_df["LC"] > 0]["LC"], bins=50)
plt.title("LC Distribution: Non-Zero Claims Only")
plt.xlabel("LC")
plt.ylabel("Count")
plt.show()

The target distribution graphs show that both LC and HALC are extremely skewed. In the original histograms, almost all of the observations are concentrated near zero, while a small number stretch far to the right. This shows that most policyholders either had no claim or very low loss cost, but a few had very large losses.

The log distributions of LC and HALC make the graphs easier to read by shrinking the effect of the extreme values. However, there is still a large spike at zero, which confirms that the data has many zero-claim policies. Even when looking only at non-zero LC values, the distribution is still right-skewed, meaning most claims are smaller but a few are much larger.

Overall, LC and HALC are not normally distributed. They are zero-heavy and right-skewed, which makes the prediction problem more difficult. Because of this, a basic regression model may not work as well, and models that can handle insurance loss data with many zeros and large outliers may be more appropriate.

##Claim Status

In [ ]:
# Claim status distribution

train_df["CS"].value_counts().plot(kind="bar")
plt.title("Claim vs No Claim")
plt.xlabel("Claim Status: 0 = No Claim, 1 = Claim")
plt.ylabel("Number of Policies")
plt.show()

zero_claim_pct = (train_df["X.16"] == 0).mean()
claim_pct = (train_df["X.16"] > 0).mean()

print("Percentage of zero claims:", zero_claim_pct)
print("Percentage of positive claims:", claim_pct)

The claim status graph shows that the dataset is very imbalanced. Most policyholders did not make a claim, while only a much smaller group had a positive claim. Based on the output, about 88.85% of policies had zero claims, and only about 11.15% had positive claims.

This is important because it shows that the model cannot just focus on overall accuracy. Since most policies are no-claim cases, a model could look accurate just by predicting no claim most of the time. For this reason, claim status should be evaluated with a metric like ROC-AUC, which is better for checking how well the model separates claim vs. no-claim policies. Overall, this confirms that the classification part of the project has a class imbalance problem.

## Correlation heatmap

In [ ]:

key_cols = [
    "LC", "HALC", "CS",
    "X.14", "X.15", "X.16", "X.17", "X.18",
    "X.23", "X.24", "X.25", "X.28",
    "policy_duration"
]

corr_matrix = train_df[key_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

The correlation heatmap shows a few important relationships in the data. First, LC and HALC are highly correlated at 0.90, which makes sense because HALC is built from LC with the historical adjustment factor. LC is also strongly correlated with X.15 at 0.89, and HALC is strongly correlated with X.15 at 0.86, which also makes sense because X.15 represents total claim cost.

Claim status, CS, has a strong relationship with X.16 at 0.76 because CS is based on whether the number of claims is greater than zero. CS also has a moderate relationship with X.18 at 0.60, which suggests that historical claim behavior is related to whether a policyholder makes a claim.

The vehicle-related variables, such as X.23, X.24, X.25, and X.28, are also strongly correlated with each other. This makes sense because vehicle power, engine size, market value, and weight are probably connected. Overall, the heatmap helped confirm the relationships between the target variables and showed that some predictors may contain overlapping information.

### Policy duration vs claim status

In [ ]:
sns.boxplot(x=train_df["CS"], y=train_df["policy_duration"])
plt.title("Policy Duration vs Claim Status")
plt.xlabel("Claim Status: 0 = No Claim, 1 = Claim")
plt.ylabel("Policy Duration")
plt.show()

The policy duration boxplot compares policies with no claim versus policies with a claim. From the graph, the no-claim group appears to have a higher median policy duration and a wider spread compared to the claim group. This means that policies without claims tend to have been active for longer, while policies with claims seem to be more concentrated at shorter durations.

There are also many outliers in both groups, especially for the no-claim group, where some policy durations are extremely high. This suggests that policy duration varies a lot across customers and may contain unusual long-term policies. Overall, the graph suggests that policy duration may have some relationship with claim status, but the large number of outliers means we should be careful about overinterpreting it

In [ ]:
# Number of claims vs LC

sns.scatterplot(x=train_df["X.16"], y=np.log1p(train_df["LC"]))
plt.title("Claims vs LC")
plt.xlabel("Number of Claims")
plt.ylabel("log(1 + LC)")
plt.show()

The claims vs LC scatterplot shows the relationship between the number of claims and loss cost. Since LC is shown as log(1 + LC), the graph is easier to read even with large loss values.

Most observations are concentrated between 1 and 5 claims, but the loss cost varies a lot within each claim count. This means that having more claims does not automatically mean the LC will be higher. Some policies with only 1 or 2 claims still have very high loss costs, while some policies with more claims have lower loss costs.

Overall, this shows that claim severity matters, not just claim frequency. The number of claims is important, but it does not fully explain LC by itself. This supports the idea that the model needs more predictors than just claim count to estimate loss cost accurately.

##Claim Rate by Key Categorical Variables

In [ ]:
categorical_vars = ["X.7", "X.13", "X.19", "X.27"]

for col in categorical_vars:
    claim_rate_summary = train_df.groupby(col)["CS"].mean().reset_index()
    claim_rate_summary.columns = [col, "claim_rate"]

    display(claim_rate_summary)

    plt.figure(figsize=(8, 5))
    sns.barplot(data=claim_rate_summary, x=col, y="claim_rate")
    plt.title(f"Claim Rate by {col}")
    plt.xlabel(col)
    plt.ylabel("Claim Rate")
    plt.xticks(rotation=45)
    plt.show()

The claim rate by categorical variables shows that some groups have noticeably higher claim rates than others. For X.7, group 1 has a slightly higher claim rate than group 0, about 12.13% compared to 10.26%. The difference is not huge, but it still suggests X.7 may have some relationship with claim likelihood.

X.13 shows a much larger difference. Group 1 has a claim rate of about 15.93%, while group 0 has a claim rate of about 8.70%. This suggests that X.13 could be an important variable for predicting whether a policyholder makes a claim.

For X.19, the claim rate changes a lot by risk type. Risk type 2 has the highest claim rate at about 14.49%, followed by risk type 3 at about 11.38%. Risk type 1 is much lower at about 4.21%, and risk type 4 has 0%, although we should be careful because that group may have very few observations.

For X.27, category D has a higher claim rate than category P, about 12.57% compared to 8.72%. Overall, these graphs show that claim likelihood is not the same across all categories. Variables like X.13, X.19, and X.27 may be useful predictors for the classification model because they separate groups with different claim risk.

##Average LC and HALC by Risk Type

In [ ]:
risk_loss_summary = train_df.groupby("X.19").agg(
    count=("CS", "size"),
    claim_rate=("CS", "mean"),
    avg_LC=("LC", "mean"),
    avg_HALC=("HALC", "mean")
).reset_index()

display(risk_loss_summary)

plt.figure(figsize=(8, 5))
sns.barplot(data=risk_loss_summary, x="X.19", y="avg_LC")
plt.title("Average LC by Risk Type")
plt.xlabel("Risk Type (X.19)")
plt.ylabel("Average LC")
plt.show()

plt.figure(figsize=(8, 5))
sns.barplot(data=risk_loss_summary, x="X.19", y="avg_HALC")
plt.title("Average HALC by Risk Type")
plt.xlabel("Risk Type (X.19)")
plt.ylabel("Average HALC")
plt.show()

The average LC and HALC by risk type show that risk type matters for loss amounts. Risk type 1 has the lowest average losses, with an average LC around 20.94 and average HALC around 27.19. Risk types 2 and 3 are much higher, with average LC around 69.51 and 67.67, and average HALC around 125.59 and 124.03.

This tells us that risk types 2 and 3 not only have higher claim rates, but also higher average loss costs when compared to risk type 1. Risk type 4 has 0 average LC and HALC, but there are only 259 records in that group, so we should be careful about reading too much into it.

Overall, from a business perspective, different risk types seem to have different expected losses, so this variable could help the model better predict LC, HALC, and claim status.

##Net Premium vs LC

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(x=train_df["X.14"], y=np.log1p(train_df["LC"]), alpha=0.4)
plt.title("Net Premium vs LC")
plt.xlabel("Net Premium (X.14)")
plt.ylabel("log(1 + LC)")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(x=np.log1p(train_df["X.14"]), y=np.log1p(train_df["LC"]), alpha=0.4)
plt.title("Log Net Premium vs Log LC")
plt.xlabel("log(1 + Net Premium)")
plt.ylabel("log(1 + LC)")
plt.show()

The net premium vs LC plots show how premium is related to loss cost. In the first graph, most policies are clustered around lower to mid-range premium values, while LC still varies a lot. There is also a clear line of points at log(1 + LC) = 0, which represents policies with zero loss. This again shows how many policies have no claim.

The log premium vs log LC graph makes the relationship easier to see. There seems to be a slight positive relationship, meaning higher premium policies may be connected with higher loss costs, but the pattern is not very strong. There is still a lot of spread, so premium alone does not fully explain LC.

Overall, this graph connects well to the insurance pricing problem. Premiums are supposed to reflect expected risk, but because many policies have zero losses and some have much larger losses, the relationship is noisy. This supports why we need a model that uses multiple predictors instead of relying only on premium.

## LC Model Construction and Prediction

Below, we use XGBoost to build a regression model for `LC`, evaluate it on a validation split, fine-tune the model, and then generate predictions for the test set. We also implement a random forest for comparison.

In [ ]:
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Copy train_df and test_tf
lc_train_df = train_df.copy()
lc_test_df = test_df.copy()

# # # Imputing nulls in column X.27 with 'Unknown'
# # lc_train_df['X.27'] = lc_train_df['X.27'].fillna('Unknown')
# # lc_test_df['X.27'] = lc_test_df['X.27'].fillna('Unknown')

# Extract dates
date_cols = ["X.2", "X.3", "X.4", "X.5", "X.6"]
for col in date_cols:
    lc_train_df[f"{col}_year"] = lc_train_df[col].dt.year
    lc_train_df[f"{col}_month"] = lc_train_df[col].dt.month
    lc_train_df[f"{col}_day"] = lc_train_df[col].dt.day

    lc_test_df[f"{col}_year"] = lc_test_df[col].dt.year
    lc_test_df[f"{col}_month"] = lc_test_df[col].dt.month
    lc_test_df[f"{col}_day"] = lc_test_df[col].dt.day

# Drop raw date columns and columns that are not available in the test set
drop_cols_train = date_cols + ["X.15", "X.16", "X.17", "X.18", "LC", "HALC", "CS"]
drop_cols_test = date_cols

X = lc_train_df.drop(columns=drop_cols_train, errors="ignore")
y = lc_train_df["LC"]

# OOS test set
X_test_final = lc_test_df.drop(columns=drop_cols_test, errors="ignore")

categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = [col for col in X.columns if col not in categorical_cols]

print("Training features shape:", X.shape)
print("Test features shape:", X_test_final.shape)
print("Categorical columns:", categorical_cols)

# # y_train, y_test, x_train, x_test = train_test_split(y, x.reshape(-1, 1), random_state=0)

In [ ]:
print(f'Proportion of observations where LC=0: {(y == 0).sum() / len(y):.2f}')

In [ ]:
X['X.27'] = X['X.27'].astype('category')
X_test_final['X.27'] = X_test_final['X.27'].astype('category')

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
import xgboost as xgb
xg_reg = xgb.XGBRegressor(objective ='reg:tweedie',
                          tweedie_variance_power=1.5,
                          colsample_bytree = 0.3,
                          learning_rate = 0.1,
                          max_depth = 5, alpha = 10,
                          n_estimators = 100,
                          enable_categorical = True)
xg_reg.fit(X_train,y_train)
preds_xgb = xg_reg.predict(X_valid)

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(preds_xgb, y_valid)
plt.xlabel("Predictions")
plt.ylabel("Actual Values (y_test)")
plt.title("Scatterplot of Predictions vs. Actual Values")
plt.show()

In [ ]:
# MSE for XGBoost and GLM
from sklearn.metrics import mean_squared_error
print(f"Mean Squared Error XGBoost: {mean_squared_error(y_valid, preds_xgb)}")

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# 1. One-Hot Encode the categorical column(s)
# We do this on the full 'X' dataset BEFORE splitting to ensure the train and valid sets get the exact same new columns.
X_encoded = pd.get_dummies(X, columns=['X.27'], drop_first=True)

# 2. Split the newly encoded data
X_train_encoded, X_valid, y_train, y_valid = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42
)

# 3. Initialize the Random Forest Regressor
# n_jobs=-1 tells it to use all your CPU cores to train faster
# random_state=42 ensures your results are reproducible
rf_reg = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)

# 4. Fit the model to the training data
print("Training Random Forest...")
rf_reg.fit(X_train_encoded, y_train)

# 5. Make predictions on the validation set
preds_rf = rf_reg.predict(X_valid)

# 6. Calculate and print the Mean Squared Error
rf_mse = mean_squared_error(y_valid, preds_rf)

print(f"XGBoost MSE:      {mean_squared_error(y_valid, preds_xgb)}")
print(f"Random Forest MSE: {rf_mse:.0f}")

In [ ]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV

# Re-split the data
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Define the parameter grid to search over.
param_grid = {
    'tweedie_variance_power': [1.1, 1.5, 1.9],  # tweak these based on your problem domain
    'colsample_bytree': [0.3, 0.5, 0.7],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'alpha': [0, 10, 20]
}

# Initialize the XGBRegressor without preset hyperparameters (except objective and n_estimators)
xg_reg = xgb.XGBRegressor(objective='reg:tweedie',
                          n_estimators=100,
                          enable_categorical=True)

# Set up the GridSearchCV object with a 5-fold cross validation.
grid_cv = GridSearchCV(estimator=xg_reg,
                       param_grid=param_grid,
                       scoring='neg_mean_squared_error',  # you can change the scoring metric as required
                       cv=5,    # number of folds in k-fold cross validation
                       verbose=1,
                       n_jobs=-1)  # use all available cores for faster processing

# Fit GridSearchCV to the training data
grid_cv.fit(X_train, y_train)

# Output the best found hyperparameters and the best score
print("Best hyperparameters:", grid_cv.best_params_)
print("Best cross-validation score (negative MSE):", grid_cv.best_score_)

# Use the best estimator found to make predictions on the test set
preds_xgb_cv = grid_cv.predict(X_valid)

real_mse = mean_squared_error(y_valid, preds_xgb_cv)
print(f"True Validation MSE: {real_mse:.0f}")

In [ ]:
plt.scatter(preds_xgb, y_valid)
plt.xlabel("Predictions")
plt.ylabel("Actual Values (y_test)")
plt.title("Scatterplot of Predictions vs. Actual Values")
plt.show()

In [ ]:
# Run cross-validation again with a more fine-tuned parameter grid.

# The "Fine" Parameter Grid
param_grid_fine = {
    'tweedie_variance_power': [1.3, 1.5, 1.7],  # Zooming in around 1.5
    'colsample_bytree': [0.7, 0.85, 1.0],       # Pushing the ceiling higher
    'learning_rate': [0.05, 0.1, 0.15],         # Zooming in around 0.1
    'max_depth': [2, 3, 4],                     # Pushing the floor lower
    'alpha': [20, 30, 40]                       # Pushing the ceiling higher
}

# Initialize the XGBRegressor without preset hyperparameters (except objective and n_estimators)
xg_reg = xgb.XGBRegressor(objective='reg:tweedie',
                          n_estimators=100,
                          enable_categorical=True)

# Set up the GridSearchCV object with a 5-fold cross validation.
grid_cv = GridSearchCV(estimator=xg_reg,
                       param_grid=param_grid_fine,
                       scoring='neg_mean_squared_error',  # you can change the scoring metric as required
                       cv=5,    # number of folds in k-fold cross validation
                       verbose=1,
                       n_jobs=-1)  # use all available cores for faster processing

# Fit GridSearchCV to the training data
grid_cv.fit(X_train, y_train)

# Output the best found hyperparameters and the best score
print("Best hyperparameters:", grid_cv.best_params_)
print("Best cross-validation score (negative MSE):", grid_cv.best_score_)

# Use the best estimator found to make predictions on the test set
preds_xgb_cv = grid_cv.predict(X_valid)

real_mse = mean_squared_error(y_valid, preds_xgb_cv)
print(f"True Validation MSE: {real_mse:.0f}")

In [ ]:
import matplotlib.pyplot as plt
import xgboost as xgb

best_lc_model = grid_cv.best_estimator_

# Set up the plot size to look good on a poster or slide
plt.rcParams["figure.figsize"] = (10, 6)

# Plot using 'gain' to show the actual reduction in error
ax = xgb.plot_importance(
    best_lc_model,
    max_num_features=10,
    importance_type='gain',  # <-- This is the crucial change
    title='Top 10 Feature Importances by GAIN (LC)',
    xlabel='Gain (Average impact on model accuracy)', # Helps the panel understand the x-axis
    show_values=False
)

for p in ax.patches:
    # Get the width of the bar (which is the Gain value)
    value = p.get_width()
    # Add text at the end of the bar, rounded to 2 decimal places
    ax.annotate(f'{value:.0f}',
                (value, p.get_y() + p.get_height() / 2),
                ha='left', va='center',
                xytext=(5, 0),
                textcoords='offset points')
plt.grid(True)
plt.show()



In [ ]:
# Get predictions from our tuned XGBoost
final_LC_preds = grid_cv.predict(X_test_final)

In [ ]:
# Put those predictions into a DataFrame
final_LC_preds = pd.DataFrame({'LC':final_LC_preds})

## HALC Model Construction and Prediction

Below, we use XGBoost to build a regression model for `HALC`, evaluate it on a validation split, fine-tune the model, and then generate predictions for the test set. We also implement a random forest for comparison.

In [ ]:
X_halc = lc_train_df.drop(columns=drop_cols_train, errors="ignore")
y_halc = lc_train_df["HALC"]

In [ ]:
print(f'Proportion of observations where HALC=0: {(y_halc == 0).sum() / len(y_halc):.2f}')

In [ ]:
X_halc['X.27'] = X_halc['X.27'].astype('category')

In [ ]:
X_train_halc, X_valid_halc, y_train_halc, y_valid_halc = train_test_split(
    X_halc,
    y_halc,
    test_size=0.2,
    random_state=42
)

In [ ]:
import xgboost as xgb
xg_reg_halc = xgb.XGBRegressor(objective ='reg:tweedie',
                          tweedie_variance_power=1.5,
                          colsample_bytree = 0.3,
                          learning_rate = 0.1,
                          max_depth = 5, alpha = 10,
                          n_estimators = 100,
                          enable_categorical = True)
xg_reg_halc.fit(X_train_halc,y_train_halc)
preds_xgb_halc = xg_reg_halc.predict(X_valid_halc)

In [ ]:
plt.scatter(preds_xgb_halc, y_valid_halc)
plt.xlabel("Predictions")
plt.ylabel("Actual Values (y_test)")
plt.title("Scatterplot of Predictions vs. Actual Values")
plt.show()

In [ ]:
print(f"Mean Squared Error XGBoost: {mean_squared_error(y_valid_halc, preds_xgb_halc)}")

In [ ]:
# 1. One-Hot Encode the categorical column(s)
# We do this on the full 'X' dataset BEFORE splitting to ensure the train and valid sets get the exact same new columns.
X_encoded = pd.get_dummies(X_halc, columns=['X.27'], drop_first=True)

# 2. Split the newly encoded data
X_train_encoded, X_valid, y_train, y_valid = train_test_split(
    X_encoded,
    y_halc,
    test_size=0.2,
    random_state=42
)

# 3. Initialize the Random Forest Regressor
# n_jobs=-1 tells it to use all your CPU cores to train faster
# random_state=42 ensures your results are reproducible
rf_reg = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)

# 4. Fit the model to the training data
print("Training Random Forest...")
rf_reg.fit(X_train_encoded, y_train)

# 5. Make predictions on the validation set
preds_rf = rf_reg.predict(X_valid)

# 6. Calculate and print the Mean Squared Error
rf_mse = mean_squared_error(y_valid, preds_rf)

print(f"XGBoost MSE:      {mean_squared_error(y_valid_halc, preds_xgb_halc)}")
print(f"Random Forest MSE: {rf_mse:.0f}")

In [ ]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV

# Re-split the data
X_train_halc, X_valid_halc, y_train_halc, y_valid_halc = train_test_split(
    X_halc,
    y_halc,
    test_size=0.2,
    random_state=42
)

# Define the parameter grid to search over.
param_grid = {
    'tweedie_variance_power': [1.1, 1.5, 1.9],  # tweak these based on your problem domain
    'colsample_bytree': [0.3, 0.5, 0.7],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'alpha': [0, 10, 20]
}

# Initialize the XGBRegressor without preset hyperparameters (except objective and n_estimators)
xg_reg = xgb.XGBRegressor(objective='reg:tweedie',
                          n_estimators=100,
                          enable_categorical=True)

# Set up the GridSearchCV object with a 5-fold cross validation.
grid_cv = GridSearchCV(estimator=xg_reg,
                       param_grid=param_grid,
                       scoring='neg_mean_squared_error',  # you can change the scoring metric as required
                       cv=5,    # number of folds in k-fold cross validation
                       verbose=1,
                       n_jobs=-1)  # use all available cores for faster processing

# Fit GridSearchCV to the training data
grid_cv.fit(X_train_halc, y_train_halc)

# Output the best found hyperparameters and the best score
print("Best hyperparameters:", grid_cv.best_params_)
print("Best cross-validation score (negative MSE):", grid_cv.best_score_)

# Use the best estimator found to make predictions on the test set
preds_xgb_cv = grid_cv.predict(X_valid_halc)

In [ ]:
real_mse = mean_squared_error(y_valid_halc, preds_xgb_cv)
print(f"True Validation MSE: {real_mse:.0f}")

In [ ]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error

# 1. The "Fine" Parameter Grid for HALC
param_grid_halc_fine = {
    'tweedie_variance_power': [1.4, 1.5, 1.6],  # Zooming in around 1.5
    'colsample_bytree': [0.4, 0.5, 0.6],        # Zooming in around 0.5
    'learning_rate': [0.05, 0.1, 0.15],         # Zooming in around 0.1
    'max_depth': [2, 3, 4],                     # Pushing the floor lower
    'alpha': [0, 2, 5]                          # Keeping regularization extremely low
}

# 2. Initialize the XGBRegressor for HALC
xg_reg_halc_fine = xgb.XGBRegressor(objective='reg:tweedie',
                                    n_estimators=100,
                                    enable_categorical=True)

# 3. Set up the new GridSearchCV object
grid_cv_halc_fine = GridSearchCV(estimator=xg_reg_halc_fine,
                                 param_grid=param_grid_halc_fine,
                                 scoring='neg_mean_squared_error',
                                 cv=5,
                                 verbose=1,
                                 n_jobs=-1)

# 4. Fit to the HALC training data
print("Starting Fine Grid Search for HALC...")
grid_cv_halc_fine.fit(X_train_halc, y_train_halc)

# 5. Output the new best results
print("\n--- ROUND 2 RESULTS (HALC) ---")
print("Best hyperparameters:", grid_cv_halc_fine.best_params_)

# 6. Predict and calculate the True Validation MSE
preds_halc_fine = grid_cv_halc_fine.predict(X_valid_halc)
halc_fine_mse = mean_squared_error(y_valid_halc, preds_halc_fine)

print(f"True Validation MSE (Fine Grid): {halc_fine_mse:.0f}")

In [ ]:
final_HALC_preds = grid_cv_halc_fine.predict(X_test_final)

In [ ]:
# Put predictions into a DataFrame
final_HALC_preds = pd.DataFrame({'HALC':final_HALC_preds})

In [ ]:
import matplotlib.pyplot as plt
import xgboost as xgb

# 1. Extract the best, fully-trained model from your grid search
best_halc_model = grid_cv_halc_fine.best_estimator_

# 2. Set up the plot size
plt.rcParams["figure.figsize"] = (10, 6)

# 3. Plot the top 10 most important features
# importance_type='weight' shows how many times a feature appears in a tree
xgb.plot_importance(best_halc_model, max_num_features=10, importance_type='weight', title='Top 10 Feature Importances (HALC)')
plt.show()



In [ ]:
import matplotlib.pyplot as plt
import xgboost as xgb

# Set up the plot size to look good on a poster or slide
plt.rcParams["figure.figsize"] = (10, 6)

# Plot using 'gain' to show the actual reduction in error
ax2 = xgb.plot_importance(
    best_halc_model,
    max_num_features=10,
    importance_type='gain',  # <-- This is the crucial change
    title='Top 10 Feature Importances by GAIN (HALC)',
    xlabel='Gain (Average impact on model accuracy)', # Helps the panel understand the x-axis
    show_values=False
)

for p in ax2.patches:
    # Get the width of the bar (which is the Gain value)
    value = p.get_width()
    # Add text at the end of the bar, rounded to 2 decimal places
    ax2.annotate(f'{value:.0f}',
                (value, p.get_y() + p.get_height() / 2),
                ha='left', va='center',
                xytext=(5, 0),
                textcoords='offset points')

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Extract importance scores into dictionaries
lc = best_lc_model.get_booster().get_score(importance_type='gain')
halc = best_halc_model.get_booster().get_score(importance_type='gain')

# 2. Combine into a DataFrame
df = pd.DataFrame([halc, lc], index=['HALC', 'LC']).T
df = df.sort_values(by='HALC', ascending=False).head(10) # Top 10 by Gain

# 3. Plotting on the same axis
ax = df.plot(kind='barh', figsize=(10, 8), width=0.8)

# 4. Formatting
ax.set_title('Comparison of Feature Importance: LC vs HALC')
ax.set_xlabel('GAIN (average decrease in MSE)')
ax.invert_yaxis() # Highest importance at the top

# 5. Rounding the labels (the fix from before)
for p in ax.patches:
    val = p.get_width()
    ax.annotate(f'{val:.0f}',
                (val, p.get_y() + p.get_height() / 2),
                ha='left', va='center', xytext=(5, 0),
                textcoords='offset points', fontsize=9)

plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## CS Model Construction and Prediction

Below, we build a classification model for `CS`, evaluate it on a validation split, and then generate predicted claim probabilities for the test set.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# # Build a modeling copy so the earlier cleaned dataframes stay unchanged
# cs_train_df = train_df.copy()
# cs_test_df = test_df.copy()

# Copying dataframes that we can impute and use for the random forest
cs_train_df_rforest = train_df.copy()
cs_test_df_rforest = test_df.copy()

# Imputing nulls in column X.27 with 'Unknown' for random forest
cs_train_df_rforest['X.27'] = cs_train_df_rforest['X.27'].fillna('Unknown')
cs_test_df_rforest['X.27'] = cs_test_df_rforest['X.27'].fillna('Unknown')

# Adding indicator column for records with missing value in column X.6
# Imputing nulls in column X.6 with arbitrary date
cs_train_df_rforest['X.6_missing'] = cs_train_df_rforest['X.6'].isna().astype(int)
cs_train_df_rforest['X.6'] = cs_train_df_rforest['X.6'].fillna(pd.to_datetime('1900-01-01'))
cs_test_df_rforest['X.6_missing'] = cs_test_df_rforest['X.6'].isna().astype(int)
cs_test_df_rforest['X.6'] = cs_test_df_rforest['X.6'].fillna(pd.to_datetime('1900-01-01'))


date_cols = ["X.2", "X.3", "X.4", "X.5", "X.6"]
for col in date_cols:
    cs_train_df_rforest[f"{col}_year"] = cs_train_df_rforest[col].dt.year
    cs_train_df_rforest[f"{col}_month"] = cs_train_df_rforest[col].dt.month
    cs_train_df_rforest[f"{col}_day"] = cs_train_df_rforest[col].dt.day

    cs_test_df_rforest[f"{col}_year"] = cs_test_df_rforest[col].dt.year
    cs_test_df_rforest[f"{col}_month"] = cs_test_df_rforest[col].dt.month
    cs_test_df_rforest[f"{col}_day"] = cs_test_df_rforest[col].dt.day

# Drop raw date columns and columns that are not available in the test set
drop_cols_train = date_cols + ["X.15", "X.16", "X.17", "X.18", "LC", "HALC", "CS"]
drop_cols_test = date_cols

X = cs_train_df_rforest.drop(columns=drop_cols_train, errors="ignore")
y = cs_train_df_rforest["CS"]
X_test_final = cs_test_df_rforest.drop(columns=drop_cols_test, errors="ignore")

categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = [col for col in X.columns if col not in categorical_cols]

print("Training features shape:", X.shape)
print("Test features shape:", X_test_final.shape)
print("Categorical columns:", categorical_cols)


In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore"))
                ]
            ),
            categorical_cols
        ),
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median"))
                ]
            ),
            numeric_cols
        )
    ]
)

cs_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

cs_model.fit(X_train, y_train)

valid_cs_prob = cs_model.predict_proba(X_valid)[:, 1]
valid_cs_pred = (valid_cs_prob >= 0.5).astype(int)

print("Validation ROC-AUC:", roc_auc_score(y_valid, valid_cs_prob))
print("\nClassification Report:")
print(classification_report(y_valid, valid_cs_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_valid, valid_cs_pred))

In [ ]:
# Refit on the full training data, then predict CS probabilities for the test set
cs_model.fit(X, y)
test_cs_prob = cs_model.predict_proba(X_test_final)[:, 1]

cs_predictions = pd.DataFrame({
    "CS": test_cs_prob
})

# output_path = "/content/drive/MyDrive/DSO530 Project/CS_predictions.csv"
# cs_predictions.to_csv(output_path, index=False)

# print("Saved CS predictions to:", output_path)
display(cs_predictions.head(10))


## XGBoost Model for CS and Comparison with Random Forest

In this section, we train an `XGBoost` classifier for `CS`, compare its validation ROC-AUC against the earlier `Random Forest` model, and then save predictions from the better-performing model.

In [ ]:
# Boosting with null imputation + flagging

from sklearn.base import clone
import subprocess
import sys

try:
    from xgboost import XGBClassifier
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost"])
    from xgboost import XGBClassifier

rf_auc = roc_auc_score(y_valid, valid_cs_prob)

xgb_preprocessor = clone(preprocessor)
X_train_xgb = xgb_preprocessor.fit_transform(X_train)
X_valid_xgb = xgb_preprocessor.transform(X_valid)

positive_count = y_train.sum()
negative_count = len(y_train) - positive_count
scale_pos_weight = negative_count / positive_count

xgb_model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train_xgb, y_train)

valid_xgb_prob = xgb_model.predict_proba(X_valid_xgb)[:, 1]
valid_xgb_pred = (valid_xgb_prob >= 0.5).astype(int)
xgb_auc = roc_auc_score(y_valid, valid_xgb_prob)

print("Random Forest Validation ROC-AUC:", rf_auc)
print("XGBoost Validation ROC-AUC:", xgb_auc)
print("\nXGBoost Classification Report:")
print(classification_report(y_valid, valid_xgb_pred))
print("XGBoost Confusion Matrix:")
print(confusion_matrix(y_valid, valid_xgb_pred))

comparison_df = pd.DataFrame({
    "model": ["Random Forest", "XGBoost"],
    "validation_roc_auc": [rf_auc, xgb_auc]
}).sort_values("validation_roc_auc", ascending=False)

display(comparison_df)


In [ ]:
# # Preparing data for XGBoost model WITHOUT imputing for NaN

# # Build a modeling copy so the earlier cleaned dataframes stay unchanged
# cs_train_df = train_df.copy()
# cs_test_df = test_df.copy()

# # Imputing nulls in column X.27 with 'Unknown' (still needed because categorical)
# cs_train_df['X.27'] = cs_train_df['X.27'].fillna('Unknown')
# cs_test_df['X.27'] = cs_test_df['X.27'].fillna('Unknown')


# date_cols = ["X.2", "X.3", "X.4", "X.5", "X.6"]
# for col in date_cols:
#     cs_train_df[f"{col}_year"] = cs_train_df[col].dt.year
#     cs_train_df[f"{col}_month"] = cs_train_df[col].dt.month
#     cs_train_df[f"{col}_day"] = cs_train_df[col].dt.day

#     cs_test_df[f"{col}_year"] = cs_test_df[col].dt.year
#     cs_test_df[f"{col}_month"] = cs_test_df[col].dt.month
#     cs_test_df[f"{col}_day"] = cs_test_df[col].dt.day

# # Drop raw date columns and columns that are not available in the test set
# drop_cols_train = date_cols + ["X.15", "X.16", "X.17", "X.18", "LC", "HALC", "CS"]
# drop_cols_test = date_cols

# X_boost = cs_train_df.drop(columns=drop_cols_train, errors="ignore")
# y_boost = cs_train_df["CS"]
# X_boost_test_final = cs_test_df.drop(columns=drop_cols_test, errors="ignore")

# categorical_cols = X_boost.select_dtypes(include=["object", "category"]).columns.tolist()
# numeric_cols = [col for col in X_boost.columns if col not in categorical_cols]

# print("Training features shape:", X.shape)
# print("Test features shape:", X_test_final.shape)
# print("Categorical columns:", categorical_cols)

In [ ]:
# X_train, X_valid, y_train, y_valid = train_test_split(
#     X_boost,
#     y_boost,
#     test_size=0.2,
#     random_state=42,
#     stratify=y
# )

# preprocessor = ColumnTransformer(
#     transformers=[
#         (
#             "cat",
#             Pipeline(
#                 steps=[
#                     ("imputer", SimpleImputer(strategy="most_frequent")),
#                     ("onehot", OneHotEncoder(handle_unknown="ignore"))
#                 ]
#             ),
#             categorical_cols
#         ),
#         (
#             "num",
#             Pipeline(
#                 steps=[
#                     ("imputer", SimpleImputer(strategy="median"))
#                 ]
#             ),
#             numeric_cols
#         )
#     ]
# )

In [ ]:
# from sklearn.base import clone
# import subprocess
# import sys

# try:
#     from xgboost import XGBClassifier
# except ImportError:
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost"])
#     from xgboost import XGBClassifier

# rf_auc = roc_auc_score(y_valid, valid_cs_prob)

# xgb_preprocessor = clone(preprocessor)
# X_train_xgb = xgb_preprocessor.fit_transform(X_train)
# X_valid_xgb = xgb_preprocessor.transform(X_valid)

# positive_count = y_train.sum()
# negative_count = len(y_train) - positive_count
# scale_pos_weight = negative_count / positive_count

# xgb_model = XGBClassifier(
#     n_estimators=400,
#     max_depth=6,
#     learning_rate=0.05,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     objective="binary:logistic",
#     eval_metric="auc",
#     scale_pos_weight=scale_pos_weight,
#     random_state=42,
#     n_jobs=-1
# )

# xgb_model.fit(X_train_xgb, y_train)

# valid_xgb_prob = xgb_model.predict_proba(X_valid_xgb)[:, 1]
# valid_xgb_pred = (valid_xgb_prob >= 0.5).astype(int)
# xgb_auc = roc_auc_score(y_valid, valid_xgb_prob)

# print("Random Forest Validation ROC-AUC:", rf_auc)
# print("XGBoost Validation ROC-AUC:", xgb_auc)
# print("\nXGBoost Classification Report:")
# print(classification_report(y_valid, valid_xgb_pred))
# print("XGBoost Confusion Matrix:")
# print(confusion_matrix(y_valid, valid_xgb_pred))

# comparison_df = pd.DataFrame({
#     "model": ["Random Forest", "XGBoost"],
#     "validation_roc_auc": [rf_auc, xgb_auc]
# }).sort_values("validation_roc_auc", ascending=False)

# display(comparison_df)


In [ ]:
# Fit XGBoost on the full training set
X_full_xgb = xgb_preprocessor.fit_transform(X)
X_test_xgb = xgb_preprocessor.transform(X_test_final)

positive_count_full = y.sum()
negative_count_full = len(y) - positive_count_full
scale_pos_weight_full = negative_count_full / positive_count_full

xgb_model_full = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=scale_pos_weight_full,
    random_state=42,
    n_jobs=-1
)

xgb_model_full.fit(X_full_xgb, y)
xgb_test_prob = xgb_model_full.predict_proba(X_test_xgb)[:, 1]

xgb_predictions = pd.DataFrame({
    "CS": xgb_test_prob
})

# xgb_output_path = "/content/drive/MyDrive/DSO530 Project/CS_predictions_xgboost.csv"
# xgb_predictions.to_csv(xgb_output_path, index=False)

if xgb_auc > rf_auc:
    best_model_name = "XGBoost"
    best_cs_predictions = xgb_predictions.copy()
else:
    best_model_name = "Random Forest"
    best_cs_predictions = cs_predictions.copy()

# best_output_path = "/content/drive/MyDrive/DSO530 Project/CS_predictions_best_model.csv"
# best_cs_predictions.to_csv(best_output_path, index=False)

# print("Saved XGBoost predictions to:", xgb_output_path)
print("Better validation model:", best_model_name)
# print("Saved best-model predictions to:", best_output_path)
display(best_cs_predictions.head(10))


## Merging all predictions



In [ ]:
final_LC_preds.head()

In [ ]:
best_cs_predictions.head()

In [ ]:
final_HALC_preds.head()

In [ ]:
# THIS IS THE FINAL PREDICTION DATAFRAME WE WILL BE SUBMITTING.
# We will need to write it to a csv.

Section_20261_16221_3_prediction = pd.concat([final_LC_preds,
                                              final_HALC_preds,
                                              best_cs_predictions], axis=1)

Section_20261_16221_3_prediction.head()

In [ ]:
Section_20261_16221_3_prediction.to_csv(
    "/content/drive/MyDrive/DSO530 Project/Section_20261_16221_3_prediction.csv",
    index=False
)